# Observability & Evaluations on a Financial AI Agent Crew

The goal of this notebook is demonstrate how to implement Observability and Evaluations on a Crew of Financial AI agents that retrieves financial data, analyzes that financial data, runs follow-up investigation and summarizes the results in a user-facing Stock Analysis Report.

This notebook will:
1. Connect to the MCP Server.
    * You'll need to run `python mcp_server_financial_tools.py` in a Terminal before running this notebook.
    * Ensure that OLLAMA_API_KEY environment variable is set from the Terminal beforehand, or the MCP Server will not run.  Get your free Ollama API Key here: https://ollama.com/settings/keys
        * For Windows PowerShell users, the command is: `$Env:OLLAMA_API_KEY = "your_api_key"`
2. Retreive and list all the available MCP Tools.
3. Define the LLM to be used by each Agent.  While the same LLM is used across Agents, I use different `temperature` settings to give different agents more creativity (i.e. higher temperature).
    * With `use_local_llm = True` it will use Ollama model locally (llama3.1).
    * With `use_local_llm = False` it will use Ollama Cloud (gpt-oss:120b-cloud).  For this choice you'll need to set `ollama_cloud_api_key` with your Ollama API Key (get it free here:  https://ollama.com/settings/keys)
4. Define the Agents to be used in the Crew.  Each Agent has a specialized ability (e.g. analyze stock prices, analyze estimates data, run web searches, write stock reports).  I've enabled `reasoning` for all Agents since these tasks are somewhat complex and need some autonomous thought.
5. Define the Tasks to be run by the Crew.  It's important that follow-on Tasks receive the `context` from the previous Task it depends on (e.g. can't search for bullish analyst reasons if it doesn't know the most bullish analysts.)
6. Define the Crew.  Since this is `sequential`, the order of Tasks matters.
7. Review the final Stock Report.
8. Examine some Spans from the tracing.  I'm particularly focused on Tool usage by the Crew of Agents.
9. Create and execute LLM-as-a-Judge Evaluators to verify whether the final Stock Report cites the most Bullish analyst and most Bearish analyst, along with their reasoning for their views.
10. (Optional) Debugging Zone to explore individual Tools, Agents and Tasks.

### Notebook Parameters

In [1]:
test_ticker = "TSLA"

# Default MCP Server URL
mcp_server_url = "http://127.0.0.1:8000/mcp" 

# Switch between Local Ollama and Ollama Cloud, for the CrewAI LLM
use_local_llm = True

# Local LLM
ollama_url = "http://localhost:11434" # default Ollama URL
ollama_model =  "ollama/llama3.1"

# Cloud LLM
ollama_cloud_url = "https://ollama.com" # For Cloud models
ollama_cloud_model =  "ollama/gpt-oss:120b-cloud"
ollama_cloud_api_key = "your_api_key"

# Arize Phoenix running locally
traces_endpoint = "http://localhost:6006/v1/traces"
ollama_judge_model = "ollama/mistral"

### Imports

In [2]:
from crewai import Agent, Crew, LLM, Process, Task
from crewai_tools import MCPServerAdapter
import datetime
from io import StringIO
from IPython.display import Markdown
from openinference.instrumentation.crewai import CrewAIInstrumentor
import pandas as pd
import phoenix as px
from phoenix.evals import ClassificationEvaluator
from phoenix.evals.llm import LLM as PhoenixLLM
from phoenix.otel import register

In [3]:
# Don't let Pandas hide any columns
pd.set_option('display.max_columns', None)

 ### Observability Setup

In [4]:
# Launch local Phoenix server (http://localhost:6006)
session = px.launch_app()

c:\Users\sanct\AppData\Local\Programs\Python\Python313\Lib\contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
c:\Users\sanct\AppData\Local\Programs\Python\Python313\Lib\contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


c:\Users\sanct\AppData\Local\Programs\Python\Python313\Lib\site-packages\phoenix\trace\dsl\query.py:837: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_attributes = pd.DataFrame.from_records(


In [5]:
# Connect Instrumentation to Phoenix server
tracer_provider = register(endpoint=traces_endpoint)

# Instrument CrewAI to send traces to Phoenix
CrewAIInstrumentor().instrument(
    skip_dep_check=True, 
    tracer_provider=tracer_provider
)

Overriding of current TracerProvider is not allowed


OpenTelemetry Tracing Details
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



### Connect to MCP Server
This is only an experimentation notebook, so you won't see a proper closing of connections later on.  If you were creating production code, you should consider `with MCPServerAdapter` statements.  But I don't want all that extra code in each notebook cell that calls the MCP Server, so I'm taking a shortcut to open the connections once and never close them.  Be warned, don't try this in production!

In [6]:
server_params = {
    "url": mcp_server_url,
    "transport": "streamable-http"
}

mcp_tools = MCPServerAdapter(server_params)

In [7]:
print(f"Available tools: {[tool.name for tool in mcp_tools.tools]}")

Available tools: ['stock_prices', 'stock_analyst_estimates', 'webpage_search', 'webpage_fetch']


### Configure LLMs
Define the LLM to be used by each Agent.  I'm using the same Ollama model, but I use different `temperature` settings to give different agents more creativity (i.e. higher temperature).

In [8]:
if use_local_llm:
    prices_llm = LLM(model=ollama_model, base_url=ollama_url, temperature=0.5)
    estimates_llm = LLM(model=ollama_model, base_url=ollama_url, temperature=0.3)
    web_search_llm = LLM(model=ollama_model, base_url=ollama_url, temperature=0.7)
    stock_report_llm = LLM(model=ollama_model, base_url=ollama_url, temperature=0.5)
else:
    # Using Ollama Cloud gives a large context window, better reasoning and faster execution
    prices_llm = LLM(model=ollama_cloud_model, base_url=ollama_cloud_url, temperature=0.5, headers={'Authorization': ollama_cloud_api_key })
    estimates_llm = LLM(model=ollama_cloud_model, base_url=ollama_cloud_url, temperature=0.3, headers={'Authorization': ollama_cloud_api_key })
    web_search_llm = LLM(model=ollama_cloud_model, base_url=ollama_cloud_url, temperature=0.7, headers={'Authorization': ollama_cloud_api_key })
    stock_report_llm = LLM(model=ollama_cloud_model, base_url=ollama_cloud_url, temperature=0.5, headers={'Authorization': ollama_cloud_api_key })

### Configure Agents
Define the Agents to be used in the Crew.  Each Agent has a specialized ability (e.g. analyze stock prices, analyze estimates data, run web searches, write stock reports).  I've enabled `reasoning` for all Agents since these tasks are somewhat complex and need some autonomous thought.

In [9]:
prices_analyst = Agent(
    role='Stock Price Data Analyst',
    goal='Fetch stock price data using the tool, then analyze the time series to identify trends, jumps, and drops.',
    backstory=(
        """You are an expert data analyst specializing in stock price data.
        You call the stock_prices tool ONCE to get data, then you perform calculations
        on that data to identify trends, the biggest price movements, and recent prices.
        You do NOT need to call the tool multiple times - one call gives you all the data you need."""
    ),
    tools=[mcp_tools.tools["stock_prices"]],
    llm=prices_llm,
    reasoning=True,
    verbose=True
)

In [10]:
estimates_analyst = Agent(
    role='Sell-Side Analyst Data Analyst',
    goal='Analyze the various Price Targets and Stock Recommendations from Sell-Side Analysts to understand the general consensus and disagreement.',
    backstory=(
        """You are an expert data analyst specializing in sell-side analyst price targets and stock recommendations.
        Your strength lies in accurately identifying the lowest and highest price targets and mapping recommendations (Underperform < Underweight < Sell < Neutral < Hold < Equal-Weight < Buy < Overweight < Outperform) to determine bearish and bullish outliers."""
    ),
    tools=[mcp_tools.tools["stock_analyst_estimates"]],
    llm=estimates_llm,
    reasoning=True,
    verbose=True,
)

In [11]:
web_search_agent = Agent(
    role="Web Research Specialist",
    goal="Efficiently search the web to retrieve relevant articles, reports, and data based on specific instructions, summarizing key insights without bias.",
    backstory="You are an expert at conducting targeted web searches across financial, business, and market topics. You adapt to any query, focusing on accuracy and relevance while using available tools to gather information.",
    tools=[mcp_tools.tools["webpage_search"], mcp_tools.tools["webpage_fetch"]], 
    llm=web_search_llm,
    reasoning=True,
    verbose=True
)

In [12]:
stock_analyst = Agent(
    role='Stock Analyst (Report Writer)',
    goal="Analyze stock data, analyst estimates, and web search results to produce a comprehensive investment report.",
    backstory="You are a seasoned financial analyst skilled at synthesizing data from multiple sources into clear, actionable reports.",
    llm=stock_report_llm,
    reasoning=True,
    verbose=True,
)

### Configure Tasks
Define the Tasks to be run by the Crew.  It's important that follow-on Tasks receive the `context` from the previous Task it depends on (e.g. can't search for bullish analyst reasons if it doesn't know the most bullish analysts.)

In [13]:
prices_task = Task(
    name="Stock Price Trends Analysis Task",
    description="""
        1. Call the stock_prices tool ONCE for ticker {ticker} to fetch price data.
        2. Parse the JSON response containing Date and Close price fields.
        3. Calculate and identify:
           - The most recent stock price (latest date's Close value)
           - The biggest single-day price jump (largest positive difference between consecutive days)
           - The biggest single-day price drop (largest negative difference between consecutive days)
           - Overall trend (is the price generally increasing, decreasing, or flat over the period?)
        4. Summarize your findings clearly.
        
        Do NOT call the tool multiple times. Call it once, then analyze the returned data.
    """,
    expected_output="A summary of stock price trends, including the most recent stock price, biggest stock price jump (with dates), biggest stock price drop (with dates), and overall stock price trend.",
    agent=prices_analyst  
)

In [14]:
estimates_task = Task(
    name="Sell-Side Analyst Consensus & Disagreement Task",
    description="""
        Fetch the latest analyst estimates for the stock ticker {ticker} using the stock_analyst_estimates tool.
        Analyze the results to identify:
        - The most bullish firm (highest currentPriceTarget) and its recommendation.
        - The most bearish firm (lowest currentPriceTarget) and its recommendation.
        - Overall consensus (e.g., average price target, majority recommendation).
        Summarize the key findings in a structured format for use in subsequent searches.
    """,
    expected_output="A summary of analyst estimates, including the most bullish firm, most bearish firm, and overall consensus.",
    agent=estimates_analyst  
)

In [15]:
price_change_search_task = Task(
    name="Price Change Search Task",
    description="""
        Perform a targeted web search to gather information on the biggest price change for {ticker}.  Make note of the [Month] of this big price change.
        Focus on recent articles or reports highlighting this big stock price change, explaining why it happened.
        Use the webpage_search tool with a specific query like '{ticker} stock price change' or similar.
        Double check that the web search results are time relevant to the date of the big price change mentioned in the context from the previous agent.
        Summarize the key findings, including sources and main reasons behind the stock price change.
    """,
    expected_output="A concise summary of why the stock price changed significantly.",
    agent=web_search_agent,
    context=[prices_task] 
)

In [16]:
bullish_search_task = Task(
    name="Bullish Analyst Search Task",
    description="""
        Perform a targeted web search to gather information on bullish analyst opinions for the stock ticker {ticker}.
        Focus on recent articles or reports highlighting high price targets, bullish ratings, positive catalysts, and optimistic forecasts.
        Use the webpage_search tool with a specific query like '{ticker} bullish analyst price target [Bullish Firm]' or similar for the most bullish firm.
        Double check that the web search results are not stale; relevant to the current month or previous month.
        Summarize the key findings, including sources and main points of optimism.
    """,
    expected_output="A concise summary of bullish analyst perspectives, including referenced articles and key insights.",
    agent=web_search_agent,
    context=[estimates_task] 
)

In [17]:
bearish_search_task = Task(
    name="Bearish Analyst Search Task",
    description="""
        Perform a targeted web search to gather information on bearish analyst opinions for the stock ticker {ticker}.
        Focus on recent articles or reports highlighting low price targets, bearish ratings, risks, concerns, and pessimistic forecasts.
        Use the webpage_search tool with a specific query like '{ticker} bearish analyst price target [Bearish Firm]' or similar for the most bearish firm.
        Double check that the web search results are not stale; relevant to the current month or previous month.
        Summarize the key findings, including sources and main points of concern.
    """,
    expected_output="A concise summary of bearish analyst perspectives, including referenced articles and key insights.",
    agent=web_search_agent,
    context=[estimates_task] 
)

In [18]:
analyze_stock_task = Task(
    name="Stock Analyst & Report Generation Task",
    description="""
        Use the price data from prices_analyst and the summaries from the price change searches.
        Use the analyst estimates from stock_analyst_estimates and the summaries from the bullish and bearish web searches.
        Produce a comprehensive stock analysis report for {ticker}.
        Include an investment thesis, supporting evidence (bullish), points of concern (bearish), and explanations of recent stock price changes.
        Provide suggestions of potential next steps for stock research.
        Create a final recommendation.
        Do NOT include any code, the task is to create a human-readable stock report.
    """,
    expected_output="A detailed stock analysis report with clear sections.",
    agent=stock_analyst, 
    context=[prices_task, price_change_search_task, estimates_task, bullish_search_task, bearish_search_task],
    markdown=True # Ensure it's formatted so a human user can read it
)

### Setup Crew
Define the Crew.  Since this is `sequential`, the order of Tasks matters.

In [19]:
stock_analysis_crew = Crew(
    agents=[prices_analyst, estimates_analyst, web_search_agent, stock_analyst],
    tasks=[prices_task, price_change_search_task, estimates_task, bullish_search_task, bearish_search_task, analyze_stock_task],
    process=Process.sequential,
    verbose=True
)

### Kickoff Analysis

In [20]:
# Record start time, to help lookup appropriate spans later
run_start_time = datetime.datetime.now(datetime.timezone.utc)

# Kickoff the Crew
result = stock_analysis_crew.kickoff(inputs={'ticker': test_ticker})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1ecd8b0f-33e6-4c7c-a502-0289072774a5                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭────────────────────────────────────────────────── Tool Usage ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Started                                                                                             │
│  Name: create_reasoning_plan                                                                                    │
│  Status: In Progress                                                                                            │
│  Tool Args: {'plan': "Here is my detailed reasoning plan for this task:\n1. Call the stock_prices tool ONCE     │
│  for ticker TSLA to fetch price data.\n2. Parse the JSON response containing Date and Close price fields.\n3.   │
│  Calculate and identify: \n   - The most recent stock price (latest date's Close value)\n   - The biggest       │
│  single-day price jump (largest positive difference between consecutive days)\n   - The biggest single-day      │
│  price drop (largest negative difference between consecutive days)\n   - Overall trend (is the price generally  │
│  increasing, decreasing, or flat over the period?)\n4. Summarize my findings clearly.\n\nI will use my          │
│  expertise as Stock Price Data Analyst to analyze the time series data and identify trends, jumps, and          │
│  drops.", 'ready': 'True'}                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── 🧠 Reasoning Plan ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Here is my detailed reasoning plan for this task:                                                              │
│  1. Call the stock_prices tool ONCE for ticker TSLA to fetch price data.                                        │
│  2. Parse the JSON response containing Date and Close price fields.                                             │
│  3. Calculate and identify:                                                                                     │
│     - The most recent stock price (latest date's Close value)                                                   │
│     - The biggest single-day price jump (largest positive difference between consecutive days)                  │
│     - The biggest single-day price drop (largest negative difference between consecutive days)                  │
│     - Overall trend (is the price generally increasing, decreasing, or flat over the period?)                   │
│  4. Summarize my findings clearly.                                                                              │
│                                                                                                                 │
│  I will use my expertise as Stock Price Data Analyst to analyze the time series data and identify trends,       │
│  jumps, and drops.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst (Report Writer)                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Use the price data from prices_analyst and the summaries from the price change searches.               │
│          Use the analyst estimates from stock_analyst_estimates and the summaries from the bullish and bearish  │
│  web searches.                                                                                                  │
│          Produce a comprehensive stock analysis report for TSLA.                                                │
│          Include an investment thesis, supporting evidence (bullish), points of concern (bearish), and          │
│  explanations of recent stock price changes.                                                                    │
│          Provide suggestions of potential next steps for stock research.                                        │
│          Create a final recommendation.                                                                         │
│          Do NOT include any code, the task is to create a human-readable stock report.                          │
│                                                                                                                 │
│                                                                                                                 │
│  Reasoning Plan:                                                                                                │
│  Detailed Plan for TSLA Stock Analysis Report                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst (Report Writer)                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Stock Analysis Report for TSLA                                                                 │
│  ==============================================                                                                 │
│                                                                                                                 │
│  ## Investment Thesis                                                                                           │
│  -------------------                                                                                            │
│                                                                                                                 │
│  Based on our analysis of stock data, analyst estimates, and web search results, we believe that Tesla (TSLA)   │
│  is a strong investment opportunity. The company's innovative products, expanding market share, and increasing  │
│  demand for electric vehicles make it an attractive choice for long-term investors.                             │
│                                                                                                                 │
│  ## Supporting Evidence (Bullish)                                                                               │
│  -----------------------------                                                                                  │
│                                                                                                                 │
│  *   **Price Target:** Our analysis of analyst estimates suggests a strong consensus on TSLA's price target,    │
│  with an average target of $442.5.                                                                              │
│  *   **Market Capitalization:** Tesla's market capitalization has been steadily increasing, reaching $997.63    │
│  billion USD, indicating growing investor confidence in the company.                                            │
│  *   **Industry Trends:** The electric vehicle (EV) market is expected to continue growing rapidly, driven by   │
│  government regulations and consumer demand for sustainable energy solutions.                                   │
│                                                                                                                 │
│  ## Points of Concern (Bearish)                                                                                 │
│  ---------------------------                                                                                    │
│                                                                                                                 │
│  *   **Competition:** Tesla faces increasing competition from established automakers and new entrants in the    │
│  EV market.                                                                                                     │
│  *   **Regulatory Risks:** Changes in government policies or regulations could impact Tesla's business model    │
│  and profitability.                                                                                             │
│  *   **Supply Chain Disruptions:** Disruptions to Tesla's supply chain, such as component shortages or          │
│  logistics issues, could affect production and revenue.                                                         │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Stock Analyst & Report Generation Task                                                                   │
│  Agent: Stock Analyst (Report Writer)                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1ecd8b0f-33e6-4c7c-a502-0289072774a5                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # Comprehensive Stock Analysis Report for TSLA                                                   │
│  ==============================================                                                                 │
│                                                                                                                 │
│  ## Investment Thesis                                                                                           │
│  -------------------                                                                                            │
│                                                                                                                 │
│  Based on our analysis of stock data, analyst estimates, and web search results, we believe that Tesla (TSLA)   │
│  is a strong investment opportunity. The company's innovative products, expanding market share, and increasing  │
│  demand for electric vehicles make it an attractive choice for long-term investors.                             │
│                                                                                                                 │
│  ## Supporting Evidence (Bullish)                                                                               │
│  -----------------------------                                                                                  │
│                                                                                                                 │
│  *   **Price Target:** Our analysis of analyst estimates suggests a strong consensus on TSLA's price target,    │
│  with an average target of $442.5.                                                                              │
│  *   **Market Capitalization:** Tesla's market capitalization has been steadily increasing, reaching $997.63    │
│  billion USD, indicating growing investor confidence in the company.                                            │
│  *   **Industry Trends:** The electric vehicle (EV) market is expected to continue growing rapidly, driven by   │
│  government regulations and consumer demand for sustainable energy solutions.                                   │
│                                                                                                                 │
│  ## Points of Concern (Bearish)                                                                                 │
│  ---------------------------                                                                                    │
│                                                                                                                 │
│  *   **Competition:** Tesla faces increasing competition from established automakers and new entrants in the    │
│  EV market.                                                                                                     │
│  *   **Regulatory Risks:** Changes in government policies or regulations could impact Tesla's business model    │
│  and profitability.                                                                                             │
│  *   **Supply Chain Disruptions:** Disruptions to Tesla's supply chain, such as component shortages or          │
│  logistics issues, could affect production and revenue

In [21]:
stock_analysis_crew.usage_metrics

UsageMetrics(total_tokens=26010, prompt_tokens=23663, cached_prompt_tokens=0, completion_tokens=2347, successful_requests=11)

### Review Results

In [22]:
Markdown(result.raw)

# Comprehensive Stock Analysis Report for TSLA
==============================================

## Investment Thesis
-------------------

Based on our analysis of stock data, analyst estimates, and web search results, we believe that Tesla (TSLA) is a strong investment opportunity. The company's innovative products, expanding market share, and increasing demand for electric vehicles make it an attractive choice for long-term investors.

## Supporting Evidence (Bullish)
-----------------------------

*   **Price Target:** Our analysis of analyst estimates suggests a strong consensus on TSLA's price target, with an average target of $442.5.
*   **Market Capitalization:** Tesla's market capitalization has been steadily increasing, reaching $997.63 billion USD, indicating growing investor confidence in the company.
*   **Industry Trends:** The electric vehicle (EV) market is expected to continue growing rapidly, driven by government regulations and consumer demand for sustainable energy solutions.

## Points of Concern (Bearish)
---------------------------

*   **Competition:** Tesla faces increasing competition from established automakers and new entrants in the EV market.
*   **Regulatory Risks:** Changes in government policies or regulations could impact Tesla's business model and profitability.
*   **Supply Chain Disruptions:** Disruptions to Tesla's supply chain, such as component shortages or logistics issues, could affect production and revenue.

## Recent Stock Price Changes
---------------------------

Our analysis of stock price data indicates that TSLA has experienced significant fluctuations in recent months. The biggest single-day price jump was from 2025-09-15 to 2025-09-16, with a jump of $11.1199951172 ($410.0400085449 to $421.6199951172). The biggest single-day price drop was from 2025-10-13 to 2025-10-14, with a drop of $6.6599938965 ($435.8999938965 to $429.2399902344).

## Potential Next Steps for Stock Research
-----------------------------------------

1.  **Monitor Industry Trends:** Continue to track the growth and adoption of electric vehicles in the market.
2.  **Analyze Company Performance:** Monitor Tesla's financial performance, including revenue growth, profitability, and cash flow management.
3.  **Evaluate Competition:** Assess the competitive landscape for TSLA, including the strategies and offerings of other EV manufacturers.

## Final Recommendation
----------------------

Based on our analysis, we recommend that investors consider adding TSLA to their portfolio as a long-term investment opportunity. However, it is essential to maintain a diversified portfolio and regularly review and adjust investments based on changing market conditions and company performance.

### Disclosure

This report has been prepared for general information purposes only and does not constitute investment advice or a recommendation to buy or sell any securities. The views expressed in this report are those of the author and do not necessarily reflect the opinions of any other person or organization.

## Evaluate the Result

In [23]:
px_client = px.Client()

### Examine Spans

In [24]:
# Examine spans that exist since experiment start time
spans_df = px_client.get_spans_dataframe()
spans_df = spans_df[spans_df['start_time'] >= run_start_time]
spans_df.head()

C:\Users\sanct\AppData\Local\Temp\ipykernel_20696\2938260727.py:2: DeprecationWarning: Migrate to client.spans.get_spans_dataframe() from arize-phoenix-client
  spans_df = px_client.get_spans_dataframe()


,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.graph.node.parent_id,attributes.task_id,attributes.crew_key,attributes.crew_id,attributes.task_key,attributes.output.mime_type,attributes.openinference.span.kind,attributes.output.value,attributes.input.value,attributes.graph.node.id,attributes.tool.name,attributes.crew_tasks,attributes.input.mime_type,attributes.crew_agents,attributes.crew_inputs
context.span_id,,,,,,,,,,,,,,,,,,,,,,,,,
c86665a3fa4f6e6b,Stock Analyst (Report Writer)._execute_core,AGENT,d46bde8e959f1f38,2025-10-18 23:15:27.417873+00:00,2025-10-18 23:15:44.435573+00:00,OK,,[],c86665a3fa4f6e6b,077b5d1731da86c1e3456a2f77d5fe0b,Web Research Specialist,b7104d17-fd9d-4b3d-840c-512d0d5e5986,7119c7bff8b868cbd3b802f848e58767,1ecd8b0f-33e6-4c7c-a502-0289072774a5,4676e489b1093d09f80b893d53b48b59,application/json,AGENT,"{""description"": ""\n Use the price data ...","{""agent"": ""id=UUID('3e034fc5-218b-498f-87d3-86...",Stock Analyst (Report Writer),None,None,None,None,None
b84068ce5fd8588e,webpage_search._use,TOOL,851f57fe04b90b64,2025-10-18 23:15:10.593439+00:00,2025-10-18 23:15:12.452918+00:00,OK,,[],b84068ce5fd8588e,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""![...","{""tool_string"": ""Action: webpage_search\nActio...",None,webpage_search,None,None,None,None
851f57fe04b90b64,Web Research Specialist._execute_core,AGENT,d46bde8e959f1f38,2025-10-18 23:15:05.772603+00:00,2025-10-18 23:15:25.316698+00:00,OK,,[],851f57fe04b90b64,077b5d1731da86c1e3456a2f77d5fe0b,Sell-Side Analyst Data Analyst,1a07abe8-57f1-4e2f-82c0-6f4fb92fbd63,7119c7bff8b868cbd3b802f848e58767,1ecd8b0f-33e6-4c7c-a502-0289072774a5,4a6631b8e310715e6f4895ecdfff9f9c,application/json,AGENT,"{""description"": ""\n Perform a targeted ...","{""agent"": ""id=UUID('f8124fc5-0518-406d-a5e7-0a...",Web Research Specialist,None,None,None,None,None
8b9d1db90e628ce1,webpage_search._use,TOOL,dadad71e52242bbb,2025-10-18 23:14:49.873463+00:00,2025-10-18 23:14:51.338377+00:00,OK,,[],8b9d1db90e628ce1,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""Ma...","{""tool_string"": ""Thought: I need to find recen...",None,webpage_search,None,None,None,None
dadad71e52242bbb,Web Research Specialist._execute_core,AGENT,d46bde8e959f1f38,2025-10-18 23:14:43.295129+00:00,2025-10-18 23:15:03.686238+00:00,OK,,[],dadad71e52242bbb,077b5d1731da86c1e3456a2f77d5fe0b,Sell-Side Analyst Data Analyst,75d2eb8a-8ca1-47be-8e98-08bafb13448a,7119c7bff8b868cbd3b802f848e58767,1ecd8b0f-33e6-4c7c-a502-0289072774a5,320fdde2696e00ceeec8bf30c9c5d1b4,application/json,AGENT,"{""description"": ""\n Perform a targeted ...","{""agent"": ""id=UUID('f8124fc5-0518-406d-a5e7-0a...",Web Research Specialist,None,None,None,None,None


In [25]:
# Examine just Tool spans
tools_df = spans_df[spans_df['span_kind'] == 'TOOL']
tools_df.head()

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.graph.node.parent_id,attributes.task_id,attributes.crew_key,attributes.crew_id,attributes.task_key,attributes.output.mime_type,attributes.openinference.span.kind,attributes.output.value,attributes.input.value,attributes.graph.node.id,attributes.tool.name,attributes.crew_tasks,attributes.input.mime_type,attributes.crew_agents,attributes.crew_inputs
context.span_id,,,,,,,,,,,,,,,,,,,,,,,,,
b84068ce5fd8588e,webpage_search._use,TOOL,851f57fe04b90b64,2025-10-18 23:15:10.593439+00:00,2025-10-18 23:15:12.452918+00:00,OK,,[],b84068ce5fd8588e,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""![...","{""tool_string"": ""Action: webpage_search\nActio...",None,webpage_search,None,None,None,None
8b9d1db90e628ce1,webpage_search._use,TOOL,dadad71e52242bbb,2025-10-18 23:14:49.873463+00:00,2025-10-18 23:14:51.338377+00:00,OK,,[],8b9d1db90e628ce1,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""Ma...","{""tool_string"": ""Thought: I need to find recen...",None,webpage_search,None,None,None,None
2e05903cff715a3f,stock_analyst_estimates._use,TOOL,71c7c05b74e9ea35,2025-10-18 23:14:27.107502+00:00,2025-10-18 23:14:27.324094+00:00,OK,,[],2e05903cff715a3f,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"[{""Firm"":""Baird"",""currentPriceTarget"":548.0,""T...","{""tool_string"": ""Action: stock_analyst_estimat...",None,stock_analyst_estimates,None,None,None,None
62d1d6b9673b332b,webpage_search._use,TOOL,db066785b05edb58,2025-10-18 23:14:08.211939+00:00,2025-10-18 23:14:09.682467+00:00,OK,,[],62d1d6b9673b332b,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""# ...","{""tool_string"": ""Action: webpage_search\nActio...",None,webpage_search,None,None,None,None
03fd6335948ce6cb,stock_prices._use,TOOL,cfc285b2ad44c09b,2025-10-18 23:13:51.201158+00:00,2025-10-18 23:13:51.337807+00:00,OK,,[],03fd6335948ce6cb,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"[{""Date"":""2025-07-18T04:00:00.000Z"",""Close"":32...","{""tool_string"": ""Action: stock_prices\nAction ...",None,stock_prices,None,None,None,None


### Figure out the most Bullish & Bearish Analysts from `stock_analyst_estimates` Tool call

In [26]:
# Get the Span for the Stock Analyst Estimates lookup tool call
# There should only be one call, as seen below
estimates_lookup_df = spans_df[spans_df['name']=='stock_analyst_estimates._use']
estimates_lookup_df.head()


,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.graph.node.parent_id,attributes.task_id,attributes.crew_key,attributes.crew_id,attributes.task_key,attributes.output.mime_type,attributes.openinference.span.kind,attributes.output.value,attributes.input.value,attributes.graph.node.id,attributes.tool.name,attributes.crew_tasks,attributes.input.mime_type,attributes.crew_agents,attributes.crew_inputs
context.span_id,,,,,,,,,,,,,,,,,,,,,,,,,
2e05903cff715a3f,stock_analyst_estimates._use,TOOL,71c7c05b74e9ea35,2025-10-18 23:14:27.107502+00:00,2025-10-18 23:14:27.324094+00:00,OK,,[],2e05903cff715a3f,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"[{""Firm"":""Baird"",""currentPriceTarget"":548.0,""T...","{""tool_string"": ""Action: stock_analyst_estimat...",None,stock_analyst_estimates,None,None,None,None


In [27]:
# See that the input used the proper Ticker for lookup
estimates_lookup_df["attributes.input.value"].iloc[0]

'{"tool_string": "Action: stock_analyst_estimates\\nAction Input: {\\"ticker\\": \\"TSLA\\"}", "tool": "CrewStructuredTool(name=\'stock_analyst_estimates\', description=\'Tool Name: stock_analyst_estimates\\nTool Arguments: {\'properties\': {\'ticker\': {\'anyOf\': [], \'description\': \'\', \'enum\': None, \'items\': None, \'properties\': {}, \'title\': \'Ticker\', \'type\': \'string\'}}, \'required\': [\'ticker\'], \'title\': \'retreive_analyst_predictions_toolArguments\', \'type\': \'object\'}\\nTool Description: Get analyst recommendations and price targets for a given stock ticker.  The JSON returned will contain Firm (a.k.a. Analyst), currentPriceTarget and ToGrade (a.k.a. Recommendation).  There will only be one forecast per Firm/Analyst and currentPriceTarget=0 records are excluded.  To avoid staleness, only estimates in the last 30 days are included.\')", "calling": "tool_name=\'stock_analyst_estimates\' arguments={\'ticker\': \'TSLA\'}"}'

In [28]:
# See the output in JSON format
estimates_tool_response = estimates_lookup_df["attributes.output.value"].iloc[0]
estimates_tool_response

'[{"Firm":"Baird","currentPriceTarget":548.0,"ToGrade":"Outperform"},{"Firm":"Barclays","currentPriceTarget":350.0,"ToGrade":"Equal-Weight"},{"Firm":"Canaccord Genuity","currentPriceTarget":490.0,"ToGrade":"Buy"},{"Firm":"Cantor Fitzgerald","currentPriceTarget":355.0,"ToGrade":"Overweight"},{"Firm":"Deutsche Bank","currentPriceTarget":435.0,"ToGrade":"Buy"},{"Firm":"Evercore ISI Group","currentPriceTarget":300.0,"ToGrade":"In-Line"},{"Firm":"Goldman Sachs","currentPriceTarget":395.0,"ToGrade":"Neutral"},{"Firm":"Melius Research","currentPriceTarget":520.0,"ToGrade":"Buy"},{"Firm":"Mizuho","currentPriceTarget":450.0,"ToGrade":"Outperform"},{"Firm":"Morgan Stanley","currentPriceTarget":410.0,"ToGrade":"Overweight"},{"Firm":"Piper Sandler","currentPriceTarget":500.0,"ToGrade":"Overweight"},{"Firm":"Stifel","currentPriceTarget":483.0,"ToGrade":"Buy"},{"Firm":"UBS","currentPriceTarget":247.0,"ToGrade":"Sell"},{"Firm":"Wedbush","currentPriceTarget":600.0,"ToGrade":"Outperform"}]'

In [29]:
# Convert the JSON to pandas for easier analysis
estimates_df = pd.read_json(StringIO(estimates_tool_response)).sort_values("currentPriceTarget")
estimates_df

,Firm,currentPriceTarget,ToGrade
12,UBS,247,Sell
5,Evercore ISI Group,300,In-Line
1,Barclays,350,Equal-Weight
3,Cantor Fitzgerald,355,Overweight
6,Goldman Sachs,395,Neutral
9,Morgan Stanley,410,Overweight
4,Deutsche Bank,435,Buy
8,Mizuho,450,Outperform
11,Stifel,483,Buy
2,Canaccord Genuity,490,Buy


In [30]:
# Figure out the most Bullish analyst (highest price target)
most_bullish_analyst = estimates_df['Firm'].iloc[-1]
most_bullish_analyst

'Wedbush'

In [31]:
# Figure out the most Bearish analyst (highest price target)
most_bearish_analyst = estimates_df['Firm'].iloc[0]
most_bearish_analyst

'UBS'

### Examine how the `webpage_search` tool was used to look reasoning behind the analyst bullish/bearish views

In [32]:
# Look at just the websearch tool calls
websearch_df = spans_df[spans_df['name']=='webpage_search._use']
websearch_df

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.graph.node.parent_id,attributes.task_id,attributes.crew_key,attributes.crew_id,attributes.task_key,attributes.output.mime_type,attributes.openinference.span.kind,attributes.output.value,attributes.input.value,attributes.graph.node.id,attributes.tool.name,attributes.crew_tasks,attributes.input.mime_type,attributes.crew_agents,attributes.crew_inputs
context.span_id,,,,,,,,,,,,,,,,,,,,,,,,,
b84068ce5fd8588e,webpage_search._use,TOOL,851f57fe04b90b64,2025-10-18 23:15:10.593439+00:00,2025-10-18 23:15:12.452918+00:00,OK,,[],b84068ce5fd8588e,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""![...","{""tool_string"": ""Action: webpage_search\nActio...",None,webpage_search,None,None,None,None
8b9d1db90e628ce1,webpage_search._use,TOOL,dadad71e52242bbb,2025-10-18 23:14:49.873463+00:00,2025-10-18 23:14:51.338377+00:00,OK,,[],8b9d1db90e628ce1,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""Ma...","{""tool_string"": ""Thought: I need to find recen...",None,webpage_search,None,None,None,None
62d1d6b9673b332b,webpage_search._use,TOOL,db066785b05edb58,2025-10-18 23:14:08.211939+00:00,2025-10-18 23:14:09.682467+00:00,OK,,[],62d1d6b9673b332b,077b5d1731da86c1e3456a2f77d5fe0b,None,None,None,None,None,text/plain,TOOL,"{\n ""results"": [\n {\n ""content"": ""# ...","{""tool_string"": ""Action: webpage_search\nActio...",None,webpage_search,None,None,None,None


In [33]:
# Examine the inputs fully.
# Look for the most bullish/bearish analyst firm being mentioned in the search
for websearch in websearch_df['attributes.input.value']:
    print(websearch)
    print('-' * 80)

{"tool_string": "Action: webpage_search\nAction Input: {\"query\": \"TSLA bearish analyst price target UBS\", \"max_results\": 10}", "tool": "CrewStructuredTool(name='webpage_search', description='Tool Name: webpage_search\nTool Arguments: {'properties': {'query': {'anyOf': [], 'description': '', 'enum': None, 'items': None, 'properties': {}, 'title': 'Query', 'type': 'string'}, 'max_results': {'anyOf': [], 'default': 5, 'description': '', 'enum': None, 'items': None, 'properties': {}, 'title': 'Max Results'}}, 'required': ['query'], 'title': 'webpage_searchArguments', 'type': 'object'}\nTool Description: Perform a web search using the query.  The response contains URL, title and content for each search result.  The title and the content are the most important parts of the webpage retrieval.  The URL is useful for citing the source.')", "calling": "tool_name='webpage_search' arguments={'query': 'TSLA bearish analyst price target UBS', 'max_results': 10}"}
------------------------------

### Detect Bearish and Bullish Segments

In [34]:
ANALYST_EVAL_TEMPLATE ="""You are evaluating a stock analysis report to determine if it properly identifies and explains the most {analyst_direction} analyst's perspective.

[BEGIN DATA]
************
[Stock Report]: {report}
************
[Expected {analyst_direction} Analyst]: {expected_analyst_name}
************
[END DATA]

Evaluate whether the stock report:
1. Mentions the {analyst_direction} analyst by firm name
2. Provides specific reasons or explanations for why this analyst is {analyst_direction}

Scoring:
- "complete": The report clearly identifies the {analyst_direction} analyst by name AND provides specific reasons for their {analyst_direction} stance
- "partial_name": The report mentions {analyst_direction} views and the analyst name, but lacks specific reasoning
- "partial_reasoning": The report mentions {analyst_direction} views and specific reasoning, but lacks the analyst name
- "missing": The report does not adequately address the {analyst_direction} analyst perspective
"""

In [35]:
# Numerical mappings to the categories
analyst_choices = {
    "missing": 0, 
    "partial_reasoning": 0.5, 
    "partial_name": 0.5,
    "complete": 1
}

In [36]:
analyst_evaluator = ClassificationEvaluator(
    name="analyst_citation",
    prompt_template=ANALYST_EVAL_TEMPLATE,
    llm=PhoenixLLM( 
        provider="litellm",
        model=ollama_judge_model,
        api_base=ollama_url
    ),
    choices=analyst_choices,
    direction="maximize"
)

In [37]:
bullish_eval = analyst_evaluator.evaluate({
    "analyst_direction": "Bullish",
    "expected_analyst_name": most_bullish_analyst,
    "report": result.raw
})
bullish_eval

[Score(name='analyst_citation', score=0.5, label='partial_name', explanation="The report provides the name of the bullish analyst as 'Wedbush', but it does not explicitly state specific reasons for their bullish stance. Therefore, the evaluation is 'partial_name'.", metadata={'model': 'ollama/mistral'}, source='llm', direction='maximize')]

In [38]:
bearish_eval = analyst_evaluator.evaluate({
    "analyst_direction": "Bearish",
    "expected_analyst_name": most_bearish_analyst,
    "report": result.raw
})
bearish_eval

[Score(name='analyst_citation', score=0.5, label='partial_name', explanation="The stock report mentions the Bearish analyst by firm name (UBS), but it does not provide specific reasons or explanations for why this analyst is Bearish. Thus, the evaluation would be 'partial_name'.", metadata={'model': 'ollama/mistral'}, source='llm', direction='maximize')]

## Debugging Zone
Try out the various components (Tools, Agents, Tasks) individually.

In [39]:
assert False, "Stop full notebook execution here.  Manually debug cells below as needed."

AssertionError: Stop full notebook execution here.  Manually debug cells below as needed.

### Debug Tools

In [ ]:
mcp_tools.tools

In [ ]:
result = mcp_tools.tools["stock_prices"].run(ticker="TSLA")
df_result = pd.read_json(StringIO(result))
df_result

In [ ]:
result = mcp_tools.tools["stock_analyst_estimates"].run(ticker="TSLA")
df_result = pd.read_json(StringIO(result))
df_result.head(25)

In [ ]:
mcp_tools.tools["webpage_search"].run(query="Wedbush Tesla price target reasons")

In [ ]:
mcp_tools.tools["webpage_fetch"].run(url="https://www.tipranks.com/news/goldman-sachs-sets-the-stage-for-tesla-stock-ahead-of-q3-delivery-numbers")

### Debug Agents

In [ ]:
prices_analyst.kickoff("What is the most recent price for TSLA?  And what date is that price for?")

In [ ]:
estimates_analyst.kickoff("Which firm is most bearish about TSLA?")

In [ ]:
web_search_agent.kickoff("Why is Wedbush so bullish on TSLA?")

### Debug Tasks

In [ ]:
prices_analyst.execute_task(prices_task, context={'ticker': 'TSLA'})

In [ ]:
estimates_analyst.execute_task(estimates_task, context={'ticker': 'TSLA'})